<a href="https://colab.research.google.com/github/mutassem-almahamid1/E-commerce-backend/blob/master/OpenAI_Whisper_ar_ytdlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl">
<h1>تفريغ الصوتيات باستخدم OpenAI Whisper</h1>

هذا الملف مبني على [هذا الملف](https://colab.research.google.com/gist/Kazuki-tam/04e85708e4fd1c4b8af180d317977f4d/whisper-mock-en.ipynb)


## 📖 كيفية الاستخدام
1. شغل خطوة "اﻹعداد".
2. اختر نمط التشغيل: يوتيوب أو ملف محلي.
  - إذا اخترت upload لتفريغ ملف تقوم برفعه بنفسك،  شغل خطوة اﻹعداد أولا ثم ارفع الملف إلى مجلد `download`.
  - إذا اخترت يوتيوب ضع رابط المقطع أو قائمة التشغيل في `user_input`
  - يمكنك تحديد بداية ونهاية قائمة التشغيل من الخيارين التاليين.
3. اختر لغة الملف.
4. اختر النموذج المستخدم في التفريغ، `large` يعطي نتائج أفضل لكن أبطأ.
5. شغل خطوة "التفريغ".
</div>

In [8]:
#@title اﻹعداد
!apt install libcublas11
!rm -f .mise.toml
!MISE_VERSION=2025.11.7 curl https://mise.run | sh
!~/.local/bin/mise --version
!~/.local/bin/mise use --global python@3.12 deno@2 -y
!~/.local/bin/mise x -- python3 -m pip install --quiet --upgrade pip wheel setuptools
!~/.local/bin/mise which python3

from pathlib import Path

# Create folders
download_folder = Path("download")
if not download_folder.exists():
    download_folder.mkdir()
output_folder = Path("output")
if not output_folder.exists():
    output_folder.mkdir()

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libcublas11 is already the newest version (11.7.4.6~11.5.1-1ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 10651  100 10651    0     0  21329      0 --:--:-- --:--:-- --:--:-- 21344
mise: installing mise...
######################################################################## 100.0%
mise: installed successfully to /root/.local/bin/mise
mise: run the following to activate mise in your shell:
echo "eval \"\$(/root/.local/bin/mise activate bash)\"" >> ~/.bashrc

mise: run `mise doctor` to verify this is set up correctly
              _                                        __              
   ____ ___  (_)_______        ___  ____        ____  / /___ _________
  / __ `__ \/ / ___/ _ \______/ _ \/ __ \____

In [10]:
import sys
import os
import shutil
sys.path.append('/root/.local/share/mise/installs/python/3.12/lib/python3.12/site-packages')

# @title التفريغ
import mimetypes
import subprocess
from pathlib import Path

from google.colab import files

# These two functions are adapted from original whisper implementation
def format_timestamp(seconds, always_include_hours=False, decimal_marker='.'):
    if seconds is not None:
      assert seconds >= 0, 'non-negative timestamp expected'
      milliseconds = round(seconds * 1000.0)
      hours = milliseconds // 3_600_000
      milliseconds -= hours * 3_600_000
      minutes = milliseconds // 60_000
      milliseconds -= minutes * 60_000
      seconds = milliseconds // 1_000
      milliseconds -= seconds * 1_000
      hours_marker = f'{hours:02d}:' if always_include_hours or hours > 0 else ''
      return f'{hours_marker}{minutes:02d}:{seconds:02d}{decimal_marker}{milliseconds:03d}'
    return seconds

def write_srt(file, segments):
    for i, segment in enumerate(segments, start=1):
        if isinstance(segment, dict):
            segment_start = segment['start']
            segment_end = segment['end']
            segment_text = segment['text']
        else:
            segment_start = segment.start
            segment_end = segment.end
            segment_text = segment.text
        start_time = format_timestamp(segment_start, always_include_hours=True, decimal_marker=',')
        end_time = format_timestamp(segment_end, always_include_hours=True, decimal_marker=',')
        file.write('%d\n' % i)
        file.write('%s --> %s\n' % (start_time, end_time))
        file.write(segment_text.strip().replace('-->', '->'))
        file.write('\n\n')

def get_media_duration_seconds(path):
    cmd = [
        'ffprobe',
        '-v',
        'error',
        '-show_entries',
        'format=duration',
        '-of',
        'default=noprint_wrappers=1:nokey=1',
        str(path),
    ]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        duration = float(result.stdout.strip())
        return duration if duration > 0 else 1.0
    except Exception:
        return 1.0

new_line_separators = {
    'White space ( )': ' ',
    'Line feed (\\n)': '\n',
    'Carriage return + Line feed (\\r\\n)': '\r\n',
    'Carriage return (\\r)': '\r'
}

# options
asr_engine = 'whisper'  # @param ['whisper', 'omnilingual']
converter = 'whisper'  # @param ['whisper', 'faster-whisper', 'whisper-jax', 'insanely-fast-whisper']
process_type = 'youtube'  # @param ['youtube', 'google_drive', 'upload', 'direct_link', 'curl']
user_input = 'https://www.youtube.com/playlist?list=PLA091A2248A2015E2'  # @param {type:'string'}
playlist_start = 1  # @param {type:'integer'}
playlist_end = 11  # @param {type:'integer'}
language = 'ar'  # @param ['en', 'ar', 'Afrikaans', 'Albanian', 'Amharic', 'Arabic', 'Armenian', 'Assamese', 'Azerbaijani', 'Bashkir', 'Basque', 'Belarusian', 'Bengali', 'Bosnian', 'Breton', 'Bulgarian', 'Burmese', 'Castilian', 'Catalan', 'Chinese', 'Croatian', 'Czech', 'Danish', 'Dutch', 'English', 'Estonian', 'Faroese', 'Finnish', 'Flemish', 'French', 'Galician', 'Georgian', 'German', 'Greek', 'Gujarati', 'Haitian', 'Haitian Creole', 'Hausa', 'Hawaiian', 'Hebrew', 'Hindi', 'Hungarian', 'Icelandic', 'Indonesian', 'Italian', 'Japanese', 'Javanese', 'Kannada', 'Kazakh', 'Khmer', 'Korean', 'Lao', 'Latin', 'Latvian', 'Letzeburgesch', 'Lingala', 'Lithuanian', 'Luxembourgish', 'Macedonian', 'Malagasy', 'Malay', 'Malayalam', 'Maltese', 'Maori', 'Marathi', 'Moldavian', 'Moldovan', 'Mongolian', 'Myanmar', 'Nepali', 'Norwegian', 'Nynorsk', 'Occitan', 'Panjabi', 'Pashto', 'Persian', 'Polish', 'Portuguese', 'Punjabi', 'Pushto', 'Romanian', 'Russian', 'Sanskrit', 'Serbian', 'Shona', 'Sindhi', 'Sinhala', 'Sinhalese', 'Slovak', 'Slovenian', 'Somali', 'Spanish', 'Sundanese', 'Swahili', 'Swedish', 'Tagalog', 'Tajik', 'Tamil', 'Tatar', 'Telugu', 'Thai', 'Tibetan', 'Turkish', 'Turkmen', 'Ukrainian', 'Urdu', 'Uzbek', 'Valencian', 'Vietnamese', 'Welsh', 'Yiddish', 'Yoruba', 'af', 'am', 'as', 'az', 'ba', 'be', 'bg', 'bn', 'bo', 'br', 'bs', 'ca', 'cs', 'cy', 'da', 'de', 'el', 'es', 'et', 'eu', 'fa', 'fi', 'fo', 'fr', 'gl', 'gu', 'ha', 'haw', 'hi', 'hr', 'ht', 'hu', 'hy', 'id', 'is', 'it', 'iw', 'ja', 'jw', 'ka', 'kk', 'km', 'kn', 'ko', 'la', 'lb', 'ln', 'lo', 'lt', 'lv', 'mg', 'mi', 'mk', 'ml', 'mn', 'mr', 'ms', 'mt', 'my', 'ne', 'nl', 'nn', 'no', 'oc', 'pa', 'pl', 'ps', 'pt', 'ro', 'ru', 'sa', 'sd', 'si', 'sk', 'sl', 'sn', 'so', 'sq', 'sr', 'su', 'sv', 'sw', 'ta', 'te', 'tg', 'th', 'tk', 'tl', 'tr', 'tt', 'uk', 'ur', 'uz', 'vi', 'yi', 'yo', 'zh']
model = 'large-v3-turbo'  # @param ['large-v3-turbo', 'distil-large-v3', 'large-v3', 'large-v2', 'medium', 'base', 'small', 'tiny']
new_line_separator = 'Line feed (\\\\n)'  # @param {type:'string'} ['White space ( )', 'Line feed (\\n)', 'Carriage return + Line feed (\\r\\n)', 'Carriage return (\\r)', ' ']

OMNI_MODEL_CARD = 'omniASR_LLM_Unlimited_3B_v2'
OMNI_LANG_CODE = 'ara_Arab'  # @param {type:'string'}
OMNI_BATCH_SIZE = 1  # @param {type:'integer'}

# init
new_line_separator = new_line_separators.get(new_line_separator, ' ')
use_omnilingual = asr_engine == 'omnilingual'
if use_omnilingual and OMNI_MODEL_CARD != 'omniASR_LLM_Unlimited_3B_v2':
    print('Phase 1 enforces omniASR_LLM_Unlimited_3B_v2. Overriding model card.')
    OMNI_MODEL_CARD = 'omniASR_LLM_Unlimited_3B_v2'

is_whisper = converter == 'whisper'
is_faster_whisper = converter == 'faster-whisper'
is_whisper_jax = converter == 'whisper-jax'
is_insanely_fast_whisper = converter == 'insanely-fast-whisper'

mimetypes.init()

# Dynamically find deno path
deno_path = shutil.which("deno") or "/root/.local/share/mise/shims/deno"

ydl_opts = {
    'js_runtimes': {
        'deno': {'path': deno_path},
    },
    'format': 'm4a/bestaudio/best',
    'postprocessors': [
        {
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'm4a',
        }
    ],
    'playliststart': int(playlist_start),
    'playlistend': int(playlist_end),
    'outtmpl': f'{str(download_folder)}/%(playlist_index)04d-%(title)s-%(id)s.%(ext)s',
}

# Download youtube videos
if process_type == 'youtube':
    !~/.local/bin/mise x -- python3 -m pip install --quiet yt-dlp[default,curl-cffi]
    import yt_dlp
    for url in user_input.split():
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download(url)
if process_type == 'google_drive':
    !wget --quiet https://github.com/tanaikech/goodls/releases/download/v2.0.3/goodls_linux_amd64 -O download/goodls && chmod +x download/goodls
    for url in user_input.split():
        !cd download && ./goodls -u $url
if process_type == 'direct_link':
    !cd download && wget $user_input
if process_type == 'curl':
    !cd download && $user_input

# prepare model
if use_omnilingual:
    # fairseq2 in omnilingual-asr currently expects torch 2.8.x
    !~/.local/bin/mise x -- python3 -m pip install --quiet --upgrade --force-reinstall --index-url https://download.pytorch.org/whl/cu128 torch==2.8.0 torchaudio==2.8.0
    !~/.local/bin/mise x -- python3 -m pip install --quiet --upgrade --force-reinstall omnilingual-asr
    from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline

    model = ASRInferencePipeline(model_card=OMNI_MODEL_CARD)
else:
    if is_faster_whisper:
        !~/.local/bin/mise x -- python3 -m pip install --quiet 'faster-whisper @ https://github.com/SYSTRAN/faster-whisper/archive/refs/heads/master.tar.gz' CTranslate2==4.5.0
        from huggingface_hub.utils import _runtime

        _runtime._is_google_colab = False
        from faster_whisper import BatchedInferencePipeline, WhisperModel

        model = BatchedInferencePipeline(model=WhisperModel(model, device='cuda', compute_type='float16'))
    elif is_whisper_jax:
        !~/.local/bin/mise x -- python3 -m pip install --quiet git+https://github.com/sanchit-gandhi/whisper-jax.git
        import jax.numpy as jnp
        from whisper_jax import FlaxWhisperPipline

        model = FlaxWhisperPipline(f'openai/whisper-{model}', dtype=jnp.float16)
    elif is_insanely_fast_whisper:
        !~/.local/bin/mise x -- python3 -m pip install insanely-fast-whisper
        import torch
        from transformers import pipeline

        model = pipeline(
            'automatic-speech-recognition',
            f'openai/whisper-{model}',
            torch_dtype=torch.float16,
            device='cuda:0',
        )
        model.model = model.model.to_bettertransformer()
    else:
        !~/.local/bin/mise x -- python3 -m pip install --quiet git+https://github.com/openai/whisper.git stable-ts
        import whisper
        from stable_whisper import modify_model

        model = whisper.load_model(model)
        modify_model(model)

# transcribe
for audio_file in sorted(download_folder.iterdir()):
    mime = mimetypes.guess_type(audio_file)[0]
    if mime is None:
        continue
    mime_type = mime.split('/')[0]
    if mime_type not in ('audio', 'video'):
        continue

    print(f'Transcription of {audio_file} will start!')
    text_file = Path(f'{output_folder}/{audio_file.stem}.txt')
    subtitle_file = Path(f'{output_folder}/{audio_file.stem}.srt')

    if use_omnilingual:
        lang = [OMNI_LANG_CODE.strip()] if OMNI_LANG_CODE.strip() else None
        transcriptions = model.transcribe([str(audio_file)], lang=lang, batch_size=max(1, int(OMNI_BATCH_SIZE)))
        transcript_text = transcriptions[0] if transcriptions else ''
        duration = get_media_duration_seconds(audio_file)
        segments_list = [{'start': 0.0, 'end': duration, 'text': transcript_text}]

        with open(str(text_file), 'w', encoding='utf-8') as txt, open(str(subtitle_file), 'w', encoding='utf-8') as srt:
            txt.write(transcript_text.strip() + new_line_separator)
            write_srt(srt, segments_list)
    elif is_whisper:
        result = model.transcribe(str(audio_file), language=language)
        with open(str(text_file), 'w', encoding='utf-8') as txt:
            # Access segments as attribute for stable-ts result object
            for segment in result.segments:
                txt.write(segment.text.strip() + new_line_separator)
        result.to_srt_vtt(str(subtitle_file))
    else:
        if is_faster_whisper:
            segments, info = model.transcribe(str(audio_file), language=language, beam_size=5, batch_size=16)
            print("Detected language '%s' with probability %f" % (info.language, info.language_probability))
            segments_list = [segment for segment in segments]
        if is_whisper_jax:
            segments = model(str(audio_file), language=language, return_timestamps=True)['chunks']
        if is_insanely_fast_whisper:
            segments = model(
                str(audio_file),
                chunk_length_s=30,
                batch_size=24,
                return_timestamps=True,
                generate_kwargs={'language': language},
            )['chunks']
        if is_whisper_jax or is_insanely_fast_whisper:
            segments_list = [
                {'start': segment['timestamp'][0], 'end': segment['timestamp'][1], 'text': segment['text']}
                for segment in segments
            ]
        with open(str(text_file), 'w', encoding='utf-8') as txt, open(str(subtitle_file), 'w', encoding='utf-8') as srt:
            write_srt(srt, segments_list)
            for segment in segments_list:
                txt.write((segment['text'] if isinstance(segment, dict) else segment.text) + new_line_separator)

    files.download(str(text_file))
    files.download(str(subtitle_file))
    audio_file.unlink()
    print('Done!')

[youtube:tab] Extracting URL: https://www.youtube.com/playlist?list=PLA091A2248A2015E2
[youtube:tab] PLA091A2248A2015E2: Downloading webpage
[youtube:tab] PLA091A2248A2015E2: Redownloading playlist API JSON with unavailable videos
[download] Downloading playlist: الدكتور عدنان إبراهيم l سلسلة التعريف بمباحث الفلسفة
[youtube:tab] Playlist الدكتور عدنان إبراهيم l سلسلة التعريف بمباحث الفلسفة: Downloading 11 items of 12
[download] Downloading item 1 of 11
[youtube] Extracting URL: https://www.youtube.com/watch?v=KDZJmMltTwk
[youtube] KDZJmMltTwk: Downloading webpage
[youtube] KDZJmMltTwk: Downloading android vr player API JSON
[youtube] KDZJmMltTwk: Downloading player 8a6e7bc4-main
[youtube] [jsc:deno] Solving JS challenges using deno
[youtube] KDZJmMltTwk: Downloading m3u8 information
[info] KDZJmMltTwk: Downloading 1 format(s): 140
[download] download/0001-الدكتور عدنان إبراهيم l سلسلة التعريف بمباحث الفلسفة - حلقة 1-KDZJmMltTwk.m4a has already been downloaded
[download] 100% of   16.20

/root/.local/share/mise/installs/python/3.12/lib/python3.12/site-packages/stable_whisper/whisper_compatibility.py:273: UserWarning: The installed version of Whisper might be incompatible.
The detected version appears to be installed from the repository which can have compatibility issues due to multiple commits sharing the same version number. It is recommended to install version 20250625 from PyPI.
To prevent errors and performance issues, install the latest compatible version: `pip install openai-whisper==20250625` 
Use `ignore_compatibility=True` to ignore this warning. Or use transcribe_minimal().
  warnings.warn(compatibility_warning)
Transcribe: 100%|██████████| 1050.22/1050.22 [01:17<00:00, 13.52sec/s]


Saved: /content/output/0001-الدكتور عدنان إبراهيم l سلسلة التعريف بمباحث الفلسفة - حلقة 1-KDZJmMltTwk.srt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done!
